# 02 — Modelos tabulares: triaje y sospecha de enfermedad

Este notebook explica el servicio `ml-triage`, sus modelos tabulares, artefactos, métricas, matrices de confusión y análisis crítico.

In [ ]:

from pathlib import Path
import json
import os
import subprocess
import textwrap
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start: Path | None = None) -> Path:
    """Busca la raíz del proyecto localizando docker-compose.yml."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "docker-compose.yml").exists():
            return candidate
    # Si el notebook se ejecuta desde notebooks/ antes de abrir el repo correctamente.
    return start


ROOT = find_project_root()
print(f"ROOT = {ROOT}")


def read_json(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}")
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_env(path: Path) -> dict:
    env = {}
    path = Path(path)
    if not path.exists():
        return env
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env


def run_cmd(cmd: str, timeout: int = 60) -> str:
    """Ejecuta un comando shell y devuelve stdout/stderr como texto."""
    print(f"$ {cmd}")
    try:
        result = subprocess.run(
            cmd,
            shell=True,
            cwd=ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        print(f"exit_code={result.returncode}")
        return (result.stdout or "") + (result.stderr or "")
    except Exception as exc:
        print(f"[error] {exc}")
        return str(exc)


def show_bar(series, title: str, ylabel: str = "valor"):
    ax = pd.Series(series).plot(kind="bar")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


## 1. Qué predice `ml-triage`

El servicio tiene dos salidas:

1. **Triaje**:
   - Alta;
   - Media;
   - Baja.

2. **Sospecha de enfermedad**:
   - cardiopatía aguda;
   - COVID sospecha;
   - neumonía sospecha;
   - gripe/resfriado;
   - gastroenteritis;
   - ictus;
   - etc.

Ambas predicciones son apoyo a decisión, no diagnóstico definitivo.

In [ ]:
triage_training_files = [
    "services/ml-triage/training/generate_dataset.py",
    "services/ml-triage/training/rules.py",
    "services/ml-triage/training/disease_rules.py",
    "services/ml-triage/training/model.py",
    "services/ml-triage/training/train.py",
    "services/ml-triage/training/train_disease.py",
    "services/ml-triage/training/evaluate.py",
    "services/ml-triage/training/evaluate_disease.py",
    "services/ml-triage/training/critical_analysis.py",
    "services/ml-triage/training/critical_analysis_disease.py",
]

pd.DataFrame([{"archivo": p, "existe": (ROOT/p).exists()} for p in triage_training_files])


## 2. Artefactos activos

Los modelos se versionan en:

```text
models/triage/
models/disease/
```

Cada carpeta versionada puede contener:

- `model.joblib`;
- `metadata.json`;
- `metrics.json`;
- `confusion_matrix.png`;
- `critical_analysis.md`.

In [ ]:
def list_model_artifacts(root: Path):
    if not root.exists():
        return pd.DataFrame()
    rows = []
    for d in sorted(root.iterdir()):
        if d.is_dir() and (d.name.startswith("tri-") or d.name.startswith("dis-")):
            rows.append({
                "version": d.name,
                "metadata": (d/"metadata.json").exists(),
                "metrics": (d/"metrics.json").exists(),
                "confusion_matrix": (d/"confusion_matrix.png").exists(),
                "critical_analysis": (d/"critical_analysis.md").exists(),
                "model": (d/"model.joblib").exists(),
            })
    return pd.DataFrame(rows)

triage_artifacts = list_model_artifacts(ROOT/"models/triage")
disease_artifacts = list_model_artifacts(ROOT/"models/disease")
print("Triage")
display(triage_artifacts)
print("Disease")
display(disease_artifacts)


## 3. Métricas tabulares

Se cargan los `metrics.json` si existen.

In [ ]:
def collect_metrics(root: Path, prefix: str):
    rows = []
    if not root.exists():
        return pd.DataFrame()
    for d in sorted(root.iterdir()):
        if d.is_dir() and d.name.startswith(prefix):
            metrics = read_json(d/"metrics.json", default={}) or {}
            metadata = read_json(d/"metadata.json", default={}) or {}
            rows.append({
                "version": d.name,
                "accuracy": metrics.get("accuracy"),
                "f1_macro": metrics.get("f1_macro"),
                "f1_weighted": metrics.get("f1_weighted"),
                "created_at": metadata.get("created_at"),
            })
    return pd.DataFrame(rows)

triage_metrics = collect_metrics(ROOT/"models/triage", "tri-")
disease_metrics = collect_metrics(ROOT/"models/disease", "dis-")
print("Triage metrics")
display(triage_metrics)
print("Disease metrics")
display(disease_metrics)


In [ ]:
for name, df in [("Triaje", triage_metrics), ("Enfermedad", disease_metrics)]:
    if not df.empty and "accuracy" in df:
        plot_df = df.set_index("version")[[c for c in ["accuracy", "f1_macro", "f1_weighted"] if c in df.columns]].dropna(axis=1, how="all")
        if not plot_df.empty:
            ax = plot_df.plot(kind="bar")
            ax.set_title(f"Métricas modelo {name}")
            ax.set_ylabel("score")
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout()
            plt.show()


## 4. Matrices de confusión

Las matrices de confusión se guardan como imágenes PNG dentro de cada artefacto.

In [ ]:
from PIL import Image

for root in [ROOT/"models/triage", ROOT/"models/disease"]:
    if not root.exists():
        continue
    for png in sorted(root.glob("*/confusion_matrix.png")):
        print(png.relative_to(ROOT))
        img = Image.open(png)
        plt.figure(figsize=(6, 5))
        plt.imshow(img)
        plt.axis("off")
        plt.show()


## 5. Interpretación clínica

Para triaje, el error más peligroso no es simétrico:

- `Alta → Baja`: riesgo alto, demora asistencial.
- `Alta → Media`: riesgo medio, prioridad menor de la necesaria.
- `Baja → Alta`: menor riesgo clínico, pero sobrecarga recursos.

Para sospecha de enfermedad:

- COVID o neumonía no detectada implica riesgo clínico/epidemiológico.
- Falsos positivos generan pruebas y ansiedad, pero suelen ser menos graves que falsos negativos críticos.

## 6. Comandos de reproducción

Entrenar modelos tabulares:

```powershell
.\scripts\train_tabular_models.ps1
```

Evaluar modelos tabulares:

```powershell
.\scripts\evaluate_tabular_models.ps1
```

Healthcheck:

```powershell
docker compose exec ml-triage curl -s http://localhost:8002/health
```